# Interactive 3-D Lightning Strike

Interactive front-end over the validated simulation pipeline (stages 1–5).
All physics lives in importable modules — this notebook only drives them:

| module | role | validation |
|---|---|---|
| `physics3d.py` | SOR electrostatics + 3-D DBM growth | stages 1–2 |
| `physics.py` | return-stroke current tree | stage 3 |
| `colorimetry.py`, `render3d.py` | spectral radiometry, pinhole camera | stage 4 |
| `temporal.py` | leader → return stroke → decay timeline | stage 5 |

**Requirements:** `numpy opencv-python numba ipywidgets` (and `jupyterlab_widgets` in Lab).

**Timeline:** `t < 0.55` stepped leader (channel appears in exact DBM growth order, multiple advancing tips); `0.55–0.63` return stroke (luminous front sweeps up the tree from the ground attachment at constant speed in tree metric); `> 0.63` exponential decay. Animation time is a documented nonuniform warp of physical time (≈20 ms leader vs ≈70 µs stroke).

In [ ]:
import os, sys, pickle
import numpy as np
import cv2

BASE = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, BASE)
from temporal import render_frame, write_video, activation_distances, cell_weights

# Load the validated strike. To grow a fresh one (different seed/size):
#   python3 run_grow3d.py        (checkpointed; ~25 min on 1 CPU core)
#   python3 validate3d_stage2.py && python3 validate3d_stage3.py
with open(os.path.join(BASE, 'leader3d.pkl'), 'rb') as f:
    res = pickle.load(f)
with open(os.path.join(BASE, 'currents3d.pkl'), 'rb') as f:
    cur = pickle.load(f)

I = cur['I'].copy(); I[0] = 1.0
rel = I / I.max()
d_act, _ = activation_distances(res['cells'], res['parent'], cur['main'], res['attach'])
print(f"{len(res['cells'])} channel cells, projected D = {res.get('Dproj', float('nan')):.3f}")

In [ ]:
# Interactive viewer -----------------------------------------------------
# Drag the time scrubber through leader -> attachment -> return stroke ->
# decay; orbit the camera to see the parallax of the 3-D channel.
import ipywidgets as W
from IPython.display import Image, display

QUALITY = {'fast (480\u00d7270)': (480, 270), 'good (960\u00d7540)': (960, 540)}

def show(t=0.63, azimuth=0.0, exposure=15.0, quality='fast (480\u00d7270)'):
    w, h = QUALITY[quality]
    wt = cell_weights(t, len(rel), rel, d_act, d_act.max())
    frame, _ = render_frame(res, cur, t, azimuth_deg=azimuth,
                            out_w=w, out_h=h, exposure=exposure, weights=wt)
    ok, buf = cv2.imencode('.png', cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    display(Image(data=buf.tobytes()))

W.interact(show,
           t=W.FloatSlider(0.63, min=0.0, max=1.0, step=0.005,
                           description='time', continuous_update=False),
           azimuth=W.FloatSlider(0.0, min=-90, max=90, step=2.5,
                                 description='azimuth \u00b0', continuous_update=False),
           exposure=W.FloatSlider(15.0, min=4, max=32, step=1,
                                  description='exposure', continuous_update=False),
           quality=W.Dropdown(options=list(QUALITY), description='quality'));

In [ ]:
# HD still export --------------------------------------------------------
t_hd, az_hd = 0.63, 0.0   # peak return stroke, front view
frame, _ = render_frame(res, cur, t_hd, azimuth_deg=az_hd,
                        out_w=1920, out_h=1080)
cv2.imwrite('lightning3d_frame.png', cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
print('saved lightning3d_frame.png')

In [ ]:
# MP4 export -------------------------------------------------------------
RENDER_VIDEO = False   # ~90 s at 960x540 on one CPU core
if RENDER_VIDEO:
    # slow orbital drift during the decay phase, fixed during the strike
    orbit = lambda t: 18.0 * max(t - 0.63, 0.0) / 0.37
    write_video(res, cur, 'lightning3d_strike.mp4', n_frames=150, fps=30,
                out_w=960, out_h=540, azimuth_fn=orbit, log=print)
    print('saved lightning3d_strike.mp4')

In [ ]:
# Run the validation suite ----------------------------------------------
import subprocess
for stage in ('1', '2', '3', '4', '5'):
    r = subprocess.run([sys.executable, os.path.join(BASE, f'validate3d_stage{stage}.py')],
                       capture_output=True, text=True)
    print(r.stdout.strip().splitlines()[-1], f'  (stage {stage})')

## GPU notes (RTX 3090)

Single-core growth is dominated by the `_sor_sweep3` stencil in `physics3d.py` — a
textbook red-black 7-point kernel. As a CuPy `RawKernel` (or `@cuda.jit`), full-grid
sweeps at 256³ take ~1 ms on the 3090: the bounding-box approximation becomes
unnecessary, regrowth drops from ~25 min to seconds, and the interactive scrubber can
drive a *live* simulation rather than a precomputed one. The rasterization loop in
`temporal.render_frame` ports the same way (or via `cv2.cuda`). Validation stages
1–5 run unchanged against a GPU backend — they only see arrays.